# Jupyter Notebook Documentation

This notebook performs various operations related to atmospheric data processing and analysis. It utilizes `numpy` for numerical operations and `xarray` for handling NetCDF files.

It sets up test and run parameters, defines file paths, and opens input NetCDF files. 

The notebook also samples fluctuations in relative humidity values, updates the dataset with these sampled values, and sets initial environmental conditions for contrail formation. 

Finally, it saves the modified dataset to new NetCDF files to be used as meteorological input files to APCEMM.


# Import file and filepaths

In [1]:
import numpy as np
import xarray as xr

In [ ]:
test_num = 1
run_num = 2

# Define the file paths
input_file_path = '/home/chinahg/GCresearch/contrailuncertainty/PCE/APCEMM_training_sets/BASE_APCEMM_met.nc'
output_file_template = '/home/chinahg/GCresearch/contrailuncertainty/PCE/APCEMM_training_sets/test_{}/APCEMM_met_validation_{}.nc'

# Open the input NetCDF file
ds = xr.open_dataset(input_file_path)

<xarray.Dataset>
Dimensions:                (altitude: 125, time: 24)
Coordinates:
  * altitude               (altitude) float64 0.0 0.1 0.2 0.3 ... 12.2 12.3 12.4
  * time                   (time) datetime64[ns] 2023-01-01 ... 2023-01-01T23...
    reference_time         datetime64[ns] ...
Data variables:
    shear                  (altitude, time) float64 ...
    stretch                (time) float64 ...
    pressure               (altitude) float64 ...
    temperature            (altitude, time) float64 ...
    w                      (altitude, time) float64 ...
    relative_humidity_ice  (altitude, time) float64 ...
Attributes:
    description:  APCEMM input dataset
    data_source:  artificial

# Define distribution parameters for meteorological variables

In [4]:
# Make this applicable to PCE by updating with sampled values and saving input files
# Start with 10 training examples
mean_Y = 0
sigma_Y = 0.5
altitudes = 125
timesteps = 24

scaling = 15
scaled_mean = 120

# Sample fluctuations in RH values
RHi_sampled = np.round(np.random.normal(mean_Y, sigma_Y, size=timesteps) * scaling + scaled_mean, 2)
RHi_sampled_matrix = np.tile(RHi_sampled, (16, 1))

# Check if RH_sampled has any zero or negative values
if np.any(RHi_sampled_matrix <= 0):
    print("RHi_sampled contains zero or negative values.")
else:
    print("RHi_sampled does not contain any zero or negative values.")

    if RHi_sampled[0] < 117:
        print("Warning: The first value of RHi_sampled is less than 117.")

RHi_sampled does not contain any zero or negative values.


# Update meteorological base file with new perturbed meteorological parameters

In [5]:
dims = ('altitude', 'time')

# Replace the 250 hPa row of ds with the sampled RH timeseries
ds['relative_humidity_ice'][90:106, :] = RHi_sampled_matrix
# ds['relative_humidity'][14, :] = RH_w

# # specify initial contrail environmental temperature
# ds['temperature'][14, :] = 218.0 #[K]

# ds['shear'] = (dims, 4.0*np.ones((altitudes, timesteps), dtype=float)) #[m/s/km] vertical wind shear
# ds['w'] = (dims, 0.0*np.ones((altitudes, timesteps), dtype=float)) #[m/s] vertical velocity

# Save new meteorological file for use in APCEMM

In [ ]:
# Save the changes to new output files
output_file_path = output_file_template.format(test_num,run_num)
ds.to_netcdf(output_file_path)

ds.close()